In [2]:

# Instalar Gurobi (si Colab no lo tiene)
!pip -q install gurobipy





[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#importaciones
import gurobipy as gp
from gurobipy import GRB
import time
import math

In [ ]:
###############################################################
# CONFIGURACIÓN DEL MODELO
###############################################################

# Parámetros principales
rounds = 1         # Número de rondas
lanes = 5           # Keccak usa 5x5 lanes
bits = 4            # z = 4

# Tiempo máximo del solver
TIME_LIMIT = 1500

print("="*80)
print("MODELO MILP PARA ENCONTRAR EL MÍNIMO DE S-BOXES ACTIVAS EN KECCAK REDUCIDO")
print("="*80)

print("Configuración:")
print(f"  Rondas:          {rounds}")
print(f"  Lanes:           {lanes}x{lanes}")
print(f"  Bits por lane:   {bits}")
print(f"  Estado total:    {lanes*lanes*bits} bits")
print(f"  S-boxes/ronda:   {lanes*bits}")
print(f"  Tiempo límite:   {TIME_LIMIT} segundos ({TIME_LIMIT/60:.1f} minutos)")
print("="*80)

In [4]:
###############################################################
# TABLA DE ROTACIONES DE KECCAK
###############################################################

rotation_offsets = [

    [0, 36, 3, 41, 18],

    [1, 44, 10, 45, 2],

    [62, 6, 43, 15, 61],

    [28, 55, 25, 21, 56],

    [27, 20, 39, 8, 14]

]

In [5]:
###############################################################
# CREAR EL MODELO MILP
###############################################################

model = gp.Model("Keccak")

# Minimizar
model.ModelSense = GRB.MINIMIZE

# Tiempo límite
model.Params.TimeLimit = TIME_LIMIT

print("Modelo creado correctamente.")

Set parameter TimeLimit to value 300
Modelo creado correctamente.


In [ ]:
###############################################################
# CREAR VARIABLES DEL MODELO
###############################################################

print("\nCreando variables del modelo...")

# ============================================================
# Estado principal
# S[r,x,y,z]
# ============================================================

S = model.addVars(
    rounds+1,
    5,5,bits,
    vtype=GRB.BINARY,
    name="S"
)

C = model.addVars(
    rounds,5,bits,
    vtype=GRB.BINARY,
    name="C"
)

D = model.addVars(
    rounds,5,bits,
    vtype=GRB.BINARY,
    name="D"
)

B = model.addVars(
    rounds,5,5,bits,
    vtype=GRB.BINARY,
    name="B"
)

Chi = model.addVars(
    rounds,5,5,bits,
    vtype=GRB.BINARY,
    name="Chi"
)

ChiActive = model.addVars(
    rounds,5,bits,
    vtype=GRB.BINARY,
    name="ChiActive"
)

ChiAND = model.addVars(
    rounds,
    5,
    5,
    4,
    vtype=GRB.BINARY,
    name="ChiAND"
)

# ============================================================
# Contador de variables temporales XOR
# ============================================================

temp_counter = 0

print("Variables creadas correctamente.")

model.update()

print(f"Variables creadas: {model.NumVars}")


Creando variables del modelo...
Variables creadas correctamente.
Variables creadas: 460


In [8]:
#!/usr/bin/env python3

import gurobipy as gp
from gurobipy import GRB
import time


rounds = 1
lanes = 5
bits = 4

TIME_LIMIT = 300


print("="*70)
print("MILP KECCAK REDUCIDO - GUROBI")
print("="*70)

print(f"Rondas: {rounds}")
print(f"Lanes: {lanes}x{lanes}")
print(f"Bits por lane: {bits}")
print(f"Estado: {lanes*lanes*bits} bits")
print("")


rotation_offsets = [
    [0,36,3,41,18],
    [1,44,10,45,2],
    [62,6,43,15,61],
    [28,55,25,21,56],
    [27,20,39,8,14]
]


model = gp.Model("Keccak_MILP")


print("Creando variables...")


S = model.addVars(
    rounds+1,
    lanes,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="S"
)


C = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="C"
)


D = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="D"
)


B = model.addVars(
    rounds,
    lanes,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="B"
)


ChiInput = model.addVars(
    rounds,
    lanes,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="ChiInput"
)


ChiActive = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="ChiActive"
)
ChiAND = model.addVars(
    rounds,
    5,
    5,
    bits,
    vtype=GRB.BINARY,
    name="ChiAND"
)

model.update()


print("Variables creadas:")
print(model.NumVars)
print("")


def xor2(a,b,out):

    model.addConstr(out <= a+b)
    model.addConstr(out >= a-b)
    model.addConstr(out >= b-a)
    model.addConstr(out <= 2-a-b)



def xor_list(values,out):

    temp = values[0]

    for i in range(1,len(values)):

        aux = model.addVar(
            vtype=GRB.BINARY
        )

        xor2(
            temp,
            values[i],
            aux
        )

        temp = aux


    model.addConstr(
        out == temp
    )



print("Agregando restricciones iniciales...")


model.addConstr(

    gp.quicksum(
        S[0,x,y,z]
        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)
    )

    >=1

)



model.addConstr(

    gp.quicksum(
        S[rounds,x,y,z]
        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)
    )

    >=1

)



for r in range(rounds):

    print(f"Procesando ronda {r}")


    for x in range(lanes):

        for z in range(bits):

            xor_list(

                [
                    S[r,x,y,z]
                    for y in range(lanes)
                ],

                C[r,x,z]

            )



    for x in range(lanes):

        for z in range(bits):

            xor2(

                C[r,(x-1)%lanes,z],

                C[r,(x+1)%lanes,(z-1)%bits],

                D[r,x,z]

            )



    for x in range(lanes):

        for y in range(lanes):

            for z in range(bits):

                xor2(

                    S[r,x,y,z],

                    D[r,x,z],

                    B[r,x,y,z]

                )



    for x in range(lanes):

        for y in range(lanes):

            xp=(x+3*y)%lanes

            yp=y

            shift=rotation_offsets[x][y]%bits


            for z in range(bits):

                zp=(z-shift)%bits


                model.addConstr(

                    ChiInput[r,xp,yp,z]

                    ==

                    B[r,x,y,zp]

                )
    for y in range(lanes):

        for z in range(bits):

            for x in range(lanes):

                model.addConstr(

                    ChiActive[r,y,z]

                    >=

                    ChiInput[r,x,y,z]

                )


                model.addConstr(

                    ChiInput[r,x,y,z]

                    <=

                    ChiActive[r,y,z]

                )


            model.addConstr(

                ChiActive[r,y,z]

                <=

                gp.quicksum(

                    ChiInput[r,x,y,z]

                    for x in range(lanes)

                )

            )

    for y in range(lanes):

        for z in range(bits):

            for x in range(lanes):

                x1 = (x+1)%lanes
                x2 = (x+2)%lanes


                model.addConstr(
                    ChiAND[r,x,y,z]
                    <=
                    ChiInput[r,x2,y,z]
                )


                model.addConstr(
                    ChiAND[r,x,y,z]
                    <=
                    1-ChiInput[r,x1,y,z]
                )


                model.addConstr(
                    ChiAND[r,x,y,z]
                    >=
                    ChiInput[r,x2,y,z]
                    -
                    ChiInput[r,x1,y,z]
                )


                xor2(
                    ChiInput[r,x,y,z],
                    ChiAND[r,x,y,z],
                    S[r+1,x,y,z]
                )

print("")
print("Restricciones completadas")
print("")


objetivo = gp.quicksum(

    ChiActive[r,y,z]

    for r in range(rounds)

    for y in range(lanes)

    for z in range(bits)

)


model.setObjective(

    objetivo,

    GRB.MINIMIZE

)



print("Resolviendo...")


model.setParam(
    "TimeLimit",
    TIME_LIMIT
)


inicio=time.time()


model.optimize()


tiempo=time.time()-inicio



print("")
print("="*70)
print("RESULTADOS")
print("="*70)


if model.Status == GRB.OPTIMAL:

    print("Estado: OPTIMAL")

elif model.Status == GRB.TIME_LIMIT:

    print("Estado: TIME LIMIT")

else:

    print("Estado:",model.Status)



if model.SolCount > 0:
    print("SolCount:", model.SolCount)
    print("ObjVal:", model.ObjVal)



    minimo=model.ObjVal


    print("")
    print(
        "MINIMO DE S-BOXES ACTIVAS:",
        minimo
    )


    print("")
    print("Por ronda:")


    for r in range(rounds):

        cuenta=0

        for y in range(lanes):

            for z in range(bits):

                if ChiActive[r,y,z].X > 0.5:

                    cuenta+=1


        print(
            f"Ronda {r}: {cuenta}"
        )


    print("")
    print("Variables activas en estado inicial:")


    for x in range(lanes):

        for y in range(lanes):

            for z in range(bits):

                if S[0,x,y,z].X > 0.5:

                    print(
                        f"S[0,{x},{y},{z}] = 1"
                    )


print("")
print(
    "Tiempo:",
    tiempo,
    "segundos"
)

print("="*70)

MILP KECCAK REDUCIDO - GUROBI
Rondas: 1
Lanes: 5x5
Bits por lane: 4
Estado: 100 bits

Creando variables...
Variables creadas:
560

Agregando restricciones iniciales...
Procesando ronda 0

Restricciones completadas

Resolviendo...
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 1842 rows, 640 columns and 5260 nonzeros (Min)
Model fingerprint: 0x34f74209
Model has 20 linear objective coefficients
Variable types: 0 continuous, 640 integer (640 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Presolve removed 640 rows and 260 columns
Presolve time: 0.04s
Presolved: 1202 rows, 380

Para 2 rondas

In [9]:
#!/usr/bin/env python3

import gurobipy as gp
from gurobipy import GRB
import time


rounds = 2
lanes = 5
bits = 4

TIME_LIMIT = 300


print("="*70)
print("MILP KECCAK REDUCIDO - GUROBI")
print("="*70)

print(f"Rondas: {rounds}")
print(f"Lanes: {lanes}x{lanes}")
print(f"Bits por lane: {bits}")
print(f"Estado: {lanes*lanes*bits} bits")
print("")


rotation_offsets = [
    [0,36,3,41,18],
    [1,44,10,45,2],
    [62,6,43,15,61],
    [28,55,25,21,56],
    [27,20,39,8,14]
]


model = gp.Model("Keccak_MILP")


print("Creando variables...")


S = model.addVars(
    rounds+1,
    lanes,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="S"
)


C = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="C"
)


D = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="D"
)


B = model.addVars(
    rounds,
    lanes,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="B"
)


ChiInput = model.addVars(
    rounds,
    lanes,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="ChiInput"
)


ChiActive = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="ChiActive"
)
ChiAND = model.addVars(
    rounds,
    5,
    5,
    bits,
    vtype=GRB.BINARY,
    name="ChiAND"
)

model.update()


print("Variables creadas:")
print(model.NumVars)
print("")


def xor2(a,b,out):

    model.addConstr(out <= a+b)
    model.addConstr(out >= a-b)
    model.addConstr(out >= b-a)
    model.addConstr(out <= 2-a-b)



def xor_list(values,out):

    temp = values[0]

    for i in range(1,len(values)):

        aux = model.addVar(
            vtype=GRB.BINARY
        )

        xor2(
            temp,
            values[i],
            aux
        )

        temp = aux


    model.addConstr(
        out == temp
    )



print("Agregando restricciones iniciales...")


model.addConstr(

    gp.quicksum(
        S[0,x,y,z]
        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)
    )

    >=1

)



model.addConstr(

    gp.quicksum(
        S[rounds,x,y,z]
        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)
    )

    >=1

)



for r in range(rounds):

    print(f"Procesando ronda {r}")


    for x in range(lanes):

        for z in range(bits):

            xor_list(

                [
                    S[r,x,y,z]
                    for y in range(lanes)
                ],

                C[r,x,z]

            )



    for x in range(lanes):

        for z in range(bits):

            xor2(

                C[r,(x-1)%lanes,z],

                C[r,(x+1)%lanes,(z-1)%bits],

                D[r,x,z]

            )



    for x in range(lanes):

        for y in range(lanes):

            for z in range(bits):

                xor2(

                    S[r,x,y,z],

                    D[r,x,z],

                    B[r,x,y,z]

                )



    for x in range(lanes):

        for y in range(lanes):

            xp=(x+3*y)%lanes

            yp=y

            shift=rotation_offsets[x][y]%bits


            for z in range(bits):

                zp=(z-shift)%bits


                model.addConstr(

                    ChiInput[r,xp,yp,z]

                    ==

                    B[r,x,y,zp]

                )
    for y in range(lanes):

        for z in range(bits):

            for x in range(lanes):

                model.addConstr(

                    ChiActive[r,y,z]

                    >=

                    ChiInput[r,x,y,z]

                )


                model.addConstr(

                    ChiInput[r,x,y,z]

                    <=

                    ChiActive[r,y,z]

                )


            model.addConstr(

                ChiActive[r,y,z]

                <=

                gp.quicksum(

                    ChiInput[r,x,y,z]

                    for x in range(lanes)

                )

            )

    for y in range(lanes):

        for z in range(bits):

            for x in range(lanes):

                x1 = (x+1)%lanes
                x2 = (x+2)%lanes


                model.addConstr(
                    ChiAND[r,x,y,z]
                    <=
                    ChiInput[r,x2,y,z]
                )


                model.addConstr(
                    ChiAND[r,x,y,z]
                    <=
                    1-ChiInput[r,x1,y,z]
                )


                model.addConstr(
                    ChiAND[r,x,y,z]
                    >=
                    ChiInput[r,x2,y,z]
                    -
                    ChiInput[r,x1,y,z]
                )


                xor2(
                    ChiInput[r,x,y,z],
                    ChiAND[r,x,y,z],
                    S[r+1,x,y,z]
                )

print("")
print("Restricciones completadas")
print("")


objetivo = gp.quicksum(

    ChiActive[r,y,z]

    for r in range(rounds)

    for y in range(lanes)

    for z in range(bits)

)


model.setObjective(

    objetivo,

    GRB.MINIMIZE

)



print("Resolviendo...")


model.setParam(
    "TimeLimit",
    TIME_LIMIT
)


inicio=time.time()


model.optimize()


tiempo=time.time()-inicio



print("")
print("="*70)
print("RESULTADOS")
print("="*70)


if model.Status == GRB.OPTIMAL:

    print("Estado: OPTIMAL")

elif model.Status == GRB.TIME_LIMIT:

    print("Estado: TIME LIMIT")

else:

    print("Estado:",model.Status)



if model.SolCount > 0:
    print("SolCount:", model.SolCount)
    print("ObjVal:", model.ObjVal)



    minimo=model.ObjVal


    print("")
    print(
        "MINIMO DE S-BOXES ACTIVAS:",
        minimo
    )


    print("")
    print("Por ronda:")


    for r in range(rounds):

        cuenta=0

        for y in range(lanes):

            for z in range(bits):

                if ChiActive[r,y,z].X > 0.5:

                    cuenta+=1


        print(
            f"Ronda {r}: {cuenta}"
        )


    print("")
    print("Variables activas en estado inicial:")


    for x in range(lanes):

        for y in range(lanes):

            for z in range(bits):

                if S[0,x,y,z].X > 0.5:

                    print(
                        f"S[0,{x},{y},{z}] = 1"
                    )


print("")
print(
    "Tiempo:",
    tiempo,
    "segundos"
)

print("="*70)

MILP KECCAK REDUCIDO - GUROBI
Rondas: 2
Lanes: 5x5
Bits por lane: 4
Estado: 100 bits

Creando variables...
Variables creadas:
1020

Agregando restricciones iniciales...
Procesando ronda 0
Procesando ronda 1

Restricciones completadas

Resolviendo...
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300



GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information